# Installation
`sagea==0.3.3` and later versions support installation in different environments, including standard Python (>=3.12) environments and lightweight installation in JupyterLite environments. In web-based environments such as JupyterLite, some features are currently unavailable due to limited support for certain dependencies. This setup is intended for demonstration purposes only. For full functionality, please install and use sagea in a standard Python environment.

In [ ]:
%pip install sagea==0.3.3

"""
When installing sagea in web-based environments such as JupyterLite, only a minimal set of basic dependencies related to numerical computation is included by default.
Therefore, additional plotting libraries or related dependencies need to be installed manually for creating figures later.
"""
%pip install matplotlib
%pip install cartopy

# Generate a spherical harmonic coefficients (SHCs) vector, and transform it into spatial domain

In [ ]:
import sagea
import numpy as np

lmax = 5
grid_space = 3
shc_value = np.zeros(((lmax + 1) ** 2,))
shc = sagea.SHC.generate.from_array(shc_value)

shc.replace("c5,2", 1, inplace=True)
shc.replace("c4,0", -2, inplace=True)

grid = shc.synthesize.to_grid(grid_space=grid_space)

# Plot spatial distribution

In [ ]:
import cartopy


grid.plot(
    coastline=False,
    projection=cartopy.crs.Orthographic(central_longitude=0, central_latitude=30),
    gridlines=True,
)

# Reverse the SHCs, and compare the new SHCs with the old

In [ ]:
shc_reverse = grid.to_SHC(lmax=lmax)

# show the distribution of initial and reversed

shc.plot.triangle(cmap="Blues", subtitles=["(a) initial"])
shc_reverse.plot.triangle(cmap="Blues", subtitles=["(b) reversed"])

# show their difference, which should be 0 theoretically
shc_diff = shc - shc_reverse
shc_diff.plot.triangle(cmap="jet", subtitles=["(a) - (b)"])

# A real data test of signal with different maximum degree/order

In [ ]:
import sagea
import pathlib
import cartopy

# Since the web-based JupyterLite environment cannot access online download services, the necessary local terrain/coastline datasets for plotting are provided separately and configured explicitly in this section.
cartopy.config["data_dir"] = "./data/cartopy_data"

# Avoid accidental cropping of images in the Jupyter environment
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches': None}

# define paths of products
path_l2 = pathlib.Path("data/GRACE_L2_GSM_Products/GSM-2_2008001-2008031_GRAC_UTCSR_BA01_0600")
path_gif48 = pathlib.Path("data/auxiliary/GIF48.60.gfc")

# load products as SHC instance, match dates information
lmax_list = [10, 20, 40, 60]

grid_space = 1

for lmax in lmax_list:
    shc = sagea.SHC.io.from_gfc(path_l2, lmax=lmax, key="GRCOF2")
    shc_gif48 = sagea.SHC.io.from_gfc(path_gif48, lmax=lmax, key="gfc")
    shc -= shc_gif48

    # harmonic synthesis into gridded EWH field
    shc.convert(from_type='Geopotential', to_type='EWH', inplace=True)  # EWH in unit [m]
    grid = shc.synthesize.to_grid(grid_space=grid_space)

    # plot grid
    grid.plot(vmin=-0.2, vmax=0.2, projection=cartopy.crs.Robinson(), title=f"Max degree/order: {lmax}")

The results show that the higher the maximum degree/order, the more detailed the signal it represents is in the spatial domain. However, due to the common related noise present in the GRACE high-order gravity field products, the corresponding north-south strip errors become more pronounced.

Therefore, filtering and other post-processing techniques need to be applied. Please see [the next notebook](/jupyterlite/lab/index.html?path=sagea/function_02_postprocessing.ipynb) for more details.